# 5. Support Vector Machine (SVM)
### Finding an Optimal Decision Boundary

**Support Vector Machines (SVM)** find the hyperplane that maximizes the margin (distance) between classes. Data points that lie closest to the decision surface are known as **Support Vectors**.

#### Key Concepts:
1. **Maximum Margin Hyperplane**:
   $$\min_{w, b} \frac{1}{2} \|w\|^2 \quad \text{subject to } y_i(w^T x_i + b) \ge 1$$
2. **Soft-Margin ($C$ parameter)**:
   Balances maximizing margin against misclassification penalty.
   - Large $C$: Lower bias, penalizes violations heavily (narrower margin).
   - Small $C$: Higher tolerance for misclassifications (wider margin).
3. **Kernel Trick** (for Non-Linear Separation):
   Projects input data into higher-dimensional space where it becomes linearly separable.
   - Linear: $K(x, z) = x^T z$
   - Polynomial: $K(x, z) = (\gamma x^T z + r)^d$
   - Radial Basis Function (RBF): $K(x, z) = \exp(-\gamma \|x - z\|^2)$


In [ ]:
# Step 1: Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay


In [ ]:
# Step 2: Breast Cancer Classification with RBF Kernel
cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, test_size=0.25, random_state=42, stratify=cancer.target
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svm = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
svm.fit(X_train_scaled, y_train)

y_pred = svm.predict(X_test_scaled)
print(f"SVM Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Number of Support Vectors: {svm.n_support_}")
print("
Classification Report:
", classification_report(y_test, y_pred, target_names=cancer.target_names))


In [ ]:
# Step 3: Visualizing Linear vs RBF Kernel on Non-linear Data (Moons)
X_m, y_m = make_moons(n_samples=200, noise=0.25, random_state=42)
X_m_scaled = StandardScaler().fit_transform(X_m)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
kernels = [('Linear Kernel', 'linear'), ('RBF Kernel', 'rbf')]

x_min, x_max = X_m_scaled[:, 0].min() - 0.5, X_m_scaled[:, 0].max() + 0.5
y_min, y_max = X_m_scaled[:, 1].min() - 0.5, X_m_scaled[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                     np.arange(y_min, y_max, 0.02))

for ax, (title, kern) in zip(axes, kernels):
    clf = SVC(kernel=kern, C=1.0, random_state=42).fit(X_m_scaled, y_m)
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.coolwarm)
    ax.scatter(X_m_scaled[:, 0], X_m_scaled[:, 1], c=y_m, cmap=plt.cm.coolwarm, edgecolors='k')
    # Highlight support vectors
    ax.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1], s=80, facecolors='none', edgecolors='k', linewidth=1.5, label='Support Vectors')
    ax.set_title(f"{title} (Acc: {clf.score(X_m_scaled, y_m):.2f})")
    ax.legend(loc='lower right')

plt.show()


### Summary
- **Strengths**: Effective in high-dimensional spaces, robust against overfitting when margin is maximized, flexible via custom kernels.
- **Weaknesses**: Computationally heavy on large datasets ($O(N^2)$ to $O(N^3)$), sensitive to feature scaling and hyperparameter tuning ($C, \gamma$).
